## Exercícios

> Retirados de [learn-python: sqlalchemy_orm-questions](https://aviadr1.github.io/learn-advanced-python/11_db_access/exercise/sqlalchemy_orm-questions.html).

#### Q1.

Baixa e extraia o arquivo compactado com o banco de dados [Chinook database](https://www.sqlitetutorial.net/sqlite-sample-database/). Salve o arquivo `chinook.db` na mesma pasta deste script.
* Link para baixar: http://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip

<img width=500 src=https://www.sqlitetutorial.net/wp-content/uploads/2015/11/sqlite-sample-database-color.jpg>


#### Q2.

Leia o código e os comentários das células a seguir para entender como acessamos os modelos ORM de um banco já existente.

In [ ]:
from sqlalchemy import create_engine, text, MetaData
from sqlalchemy.orm import Session

engine = create_engine("sqlite+pysqlite:///chinook.db", echo=False)

### extrai as classes da base de dados Chinook
metadata = MetaData()
metadata.reflect(engine)

# O metadata tem informações sobre as tabelas
# que serão usadas para criar os modelos ORM
for table_name, table in metadata.tables.items():
    print(table_name)
    print(table.columns.keys())
    print(table.columns.items())
    print('-'*25)

### configura o objeto Base mapeando os modelos ORM das tabelas
from sqlalchemy.ext.automap import automap_base
Base = automap_base(metadata=metadata)
Base.prepare()

# o objeto Base tem os modelos ORM que podemos usar
# para manipular o banco de dados
print(Base.classes.items())

In [ ]:
# A seguir um exemplo de query na tabela Albums
# usamos o objeto Base para acessar cada modelo ORM.

session = Session(engine)
res = session.scalars(select(Base.classes.albums))
first_album = res.first()
print(first_album.AlbumId, first_album.Title)

#### Q3. 
Com base nos códigos anteriores realize as operações solicitadas nas células a seguir:


In [ ]:
### Imprima os três primeiros registros da tabela tracks
from sqlalchemy import select

stmt = select(Base.classes.tracks).limit(3)
res = session.scalars(stmt)

for track in res:
    print(track.TrackId, track.Name)

In [ ]:
### Imprima o nome da faixa e o título do álbum das primeiras 20 faixas na tabela tracks.
from sqlalchemy import select

# Para facilitar a leitura, vamos atribuir as classes a variáveis
Tracks = Base.classes.tracks
Albums = Base.classes.albums

# Construindo a query com JOIN e limitando a 20 resultados
stmt = select(Tracks.Name, Albums.Title)\
    .join(Albums, Tracks.AlbumId == Albums.AlbumId)\
    .limit(20)

# Como não estamos retornando um objeto ORM completo, mas sim colunas específicas,
# usamos session.execute() em vez de session.scalars()
resultados = session.execute(stmt)

print(f"{'Fita / Faixa':<40} | {'Álbum'}")
print("-" * 70)
for nome_faixa, titulo_album in resultados:
    print(f"{nome_faixa[:38]:<40} | {titulo_album}")

In [ ]:
### Imprima as 10 primeiras vendas de faixas da tabela invoice_items
### Para essas 10 primeiras vendas, imprima os nomes das faixas vendidas e a quantidade vendida.
from sqlalchemy import select

# Atribuindo as classes às variáveis para facilitar a leitura
InvoiceItems = Base.classes.invoice_items
Tracks = Base.classes.tracks

# ====================================================================
# 1. Imprimir as 10 primeiras vendas da tabela invoice_items
# ====================================================================
print("--- As 10 primeiras vendas (InvoiceItems) ---")
stmt1 = select(InvoiceItems).limit(10)
resultados_vendas = session.scalars(stmt1)

for venda in resultados_vendas:
    print(f"Item ID: {venda.InvoiceLineId} | Fatura ID: {venda.InvoiceId} | "
          f"Faixa ID: {venda.TrackId} | Preço: {venda.UnitPrice} | Qtd: {venda.Quantity}")

print("\n") # Quebra de linha para separar os resultados

# ====================================================================
# 2. Imprimir os nomes das faixas e a quantidade para essas 10 vendas
# ====================================================================
print("--- Nomes das faixas e quantidades vendidas ---")
# Fazendo o JOIN entre invoice_items e tracks usando a chave TrackId
stmt2 = select(Tracks.Name, InvoiceItems.Quantity)\
    .join(Tracks, InvoiceItems.TrackId == Tracks.TrackId)\
    .limit(10)

resultados_detalhados = session.execute(stmt2)

print(f"{'Nome da Faixa':<40} | {'Quantidade'}")
print("-" * 55)
for nome_faixa, quantidade in resultados_detalhados:
    print(f"{nome_faixa[:38]:<40} | {quantidade}")


In [ ]:
### Imprima os nomes das 10 faixas mais vendidas e quantas vezes foram vendidas.
from sqlalchemy import select, func

# Atribuindo as classes às variáveis
Tracks = Base.classes.tracks
InvoiceItems = Base.classes.invoice_items

# Construindo a query
stmt = select(
    Tracks.Name, 
    func.sum(InvoiceItems.Quantity).label('total_vendido')
).join(
    InvoiceItems, Tracks.TrackId == InvoiceItems.TrackId
).group_by(
    Tracks.TrackId # Agrupamos pelo ID para garantir precisão
).order_by(
    func.sum(InvoiceItems.Quantity).desc() # Ordenamos pela soma de forma decrescente
).limit(10)

# Executando a consulta
resultados_top10 = session.execute(stmt)

# Imprimindo os resultados formatados
print(f"{'Nome da Faixa':<40} | {'Total Vendido'}")
print("-" * 55)
for nome_faixa, total_vendido in resultados_top10:
    print(f"{nome_faixa[:38]:<40} | {total_vendido}")


In [ ]:
### Quem são os 10 artistas que mais venderam?
### dica: você precisa juntar as tabelas invoice_items, tracks, albums e artists
from sqlalchemy import select, func

# Atribuindo as classes às variáveis para facilitar a leitura
Artists = Base.classes.artists
Albums = Base.classes.albums
Tracks = Base.classes.tracks
InvoiceItems = Base.classes.invoice_items

# Construindo a query com múltiplos JOINs
stmt = select(
    Artists.Name, 
    func.sum(InvoiceItems.Quantity).label('total_vendido')
).join(
    Albums, Artists.ArtistId == Albums.ArtistId
).join(
    Tracks, Albums.AlbumId == Tracks.AlbumId
).join(
    InvoiceItems, Tracks.TrackId == InvoiceItems.TrackId
).group_by(
    Artists.ArtistId # Agrupamos pelo ID para evitar problemas com artistas homônimos
).order_by(
    func.sum(InvoiceItems.Quantity).desc() # Ordenamos pelas vendas decrescentes
).limit(10)

# Executando a consulta
resultados_top10_artistas = session.execute(stmt)

# Imprimindo os resultados formatados
print(f"{'Nome do Artista':<40} | {'Faixas Vendidas'}")
print("-" * 58)
for nome_artista, total_vendido in resultados_top10_artistas:
    print(f"{nome_artista[:38]:<40} | {total_vendido}")
